In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
    driver.find_element(By.ID, "password").send_keys("12345678")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Open Notifications via the real sidebar (label verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Notifications')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//*[text()='Notifications']")))
    print("PASS: Notifications page opened")
    wait.until(EC.invisibility_of_element_located((By.XPATH, "//*[text()='Loading notifications...']")))
    time.sleep(2)

    # Real controls (verified in NotificationsPage.jsx): All/Unread filters, Mark all read
    assert driver.find_element(By.XPATH, "//button[text()='All']").is_displayed()
    assert driver.find_element(By.XPATH, "//button[text()='Unread']").is_displayed()
    assert driver.find_element(By.XPATH, "//button[contains(., 'Mark all read')]").is_displayed()
    print("PASS: Notifications controls verified (All/Unread/Mark all read)")

    body = driver.find_element(By.TAG_NAME, "body").text
    if "No notifications" in body:
        print("Empty state shown: No notifications")
    else:
        # Open a notification is read-only; test mark-as-read on the first unread item
        unread_btns = [b for b in driver.find_elements(By.XPATH, "//button[@title='Mark as read']") if b.is_displayed()]
        if unread_btns:
            unread_btns[0].click()
            time.sleep(2)
            remaining = [b for b in driver.find_elements(By.XPATH, "//button[@title='Mark as read']") if b.is_displayed()]
            assert len(remaining) == len(unread_btns) - 1, "Unread count did not decrease after marking read."
            print("PASS: Notification marked as read, unread count decreased")
        else:
            print("All notifications already read; nothing to mark.")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("38_notifications_FAIL.png")
finally:
    driver.quit()